# FF49 — Paired multi-seed robustness

Dataset: **`ken_french_49`**, **49 assets**. This transfers an existing KF12 experiment;
it does not assume KF12 outcomes or tune new methods. FF49 is the primary high-dimensional comparison. Its complete-case coverage begins in July 1969; the verified archive has 7,648 training and 1,256 validation windows.

The saved `configs/cde_ff49.yaml` protocol remains unchanged: requested sample **1958–2023**,
training through **1999-12-31**, validation **2000–2004**, lookback **60**, horizon **1**, seed **42**.
Arithmetic decimal returns minus daily French RF; complete-case row exclusion without imputation;
seven-day calendar-gap segmentation and train-only standardization. **No test evaluation.**

**Prerequisite:** completed, consistent seed-42 CDE at `/kaggle/working/ff49_cde_clean`
and completed Vanilla at `/kaggle/working/ff49_vanilla`.
Both references also need their saved validation metrics.
Finish those runs in their own notebooks; this notebook never trains a missing reference.

Work through individual cells. Extraction/training/evaluation and the final upload are manual steps;
**Run All can be expensive and uploads artifacts**. Outputs are isolated at `/kaggle/working/ff49_multiseed` and
`high_dimensional/ff49_multiseed` in the central Kaggle dataset. No experiment results are populated here.

## Setup

Enable Internet and a GPU in Kaggle settings. This is the established KF12 GitHub sync cell;
`BRANCH` selects the checkout containing these notebooks. Use `main` after merging.

In [ ]:
import os
import sys

REPO_PATH = "/kaggle/working/diffusion-portfolio"
USERNAME = "Amir-Entezari"
REPO = "diffusion-portfolio"
BRANCH = "feature/high-dimensional-kaggle"

repo_url = f"https://github.com/{USERNAME}/{REPO}.git"

# Clone the specified branch if the repository doesn't exist.
if not os.path.exists(os.path.join(REPO_PATH, ".git")):
    !git clone --branch {BRANCH} --single-branch {repo_url} {REPO_PATH}

else:
    # Fetch latest changes from the specified branch.
    !git -C {REPO_PATH} fetch origin {BRANCH}

    # Switch to the branch and force-sync with remote.
    !git -C {REPO_PATH} checkout -B {BRANCH} FETCH_HEAD
    !git -C {REPO_PATH} reset --hard FETCH_HEAD

os.chdir(REPO_PATH)

SRC_PATH = os.path.join(REPO_PATH, "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# Verify the active branch and latest commit.
!git branch --show-current
!git log -1 --oneline

print(f"Repository synced with origin/{BRANCH}.")

In [ ]:
!pip install -q -e '.[dev,cde]'

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Lightweight regression tests

In [ ]:
!pytest -q tests/semantic tests/unit/test_multiseed.py tests/unit/test_evidential_regimes.py \
    tests/unit/test_routing.py tests/unit/test_spd_features.py tests/unit/test_topology.py \
    tests/unit/test_precomputed_conditioning.py tests/unit/test_highdim_research.py
assert _exit_code == 0, "Regression tests failed."

## Restore prerequisites and this experiment

Only the listed reference runs and this notebook's own namespace are restored. Local folders
are retained. Missing references fail the next check; there is no KF12 fallback.

In [ ]:
from pathlib import Path
import json
import shutil
import numpy as np
import pandas as pd
import torch
import kagglehub
from diffusion_portfolio.utils.research_checks import (
    check_checkpoint_pair, require_completed_run, source_identity,
)

HANDLE = "amirentezari/diffusion-portfolio"
EXPERIMENT = "high_dimensional/ff49_multiseed"
ROOT = Path("/kaggle/working/ff49_multiseed")
CDE_DIR = Path("/kaggle/working/ff49_cde_clean")
REFERENCES = {Path(path): config for path, config in {'/kaggle/working/ff49_cde_clean': 'configs/cde_ff49.yaml', '/kaggle/working/ff49_vanilla': 'configs/vanilla_ff49.yaml'}.items()}

In [ ]:
artifact_path = Path(kagglehub.dataset_download(HANDLE))
restores = {"high_dimensional/" + path.name: path for path in REFERENCES}
restores[EXPERIMENT] = ROOT
for relative, destination in restores.items():
    source = artifact_path / relative
    if destination.exists():
        print("Already available:", destination)
    elif source.is_dir():
        if destination in REFERENCES:
            require_completed_run(source, REFERENCES[destination], assets=49, validation=True)
        shutil.copytree(source, destination)
        print("Restored:", destination)
    else:
        print("No backup:", relative)

In [ ]:
for path, config in REFERENCES.items():
    require_completed_run(path, config, assets=49, validation=True)
cfg = require_completed_run(CDE_DIR, "configs/cde_ff49.yaml", assets=49)
SOURCE = source_identity(CDE_DIR)
print("Prerequisites complete:", cfg.data.dataset, "assets:", 49, "seed:", cfg.seed)

## Data check

Counts and continuity are checked by the existing data tool; this does not evaluate forecasts.

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/data_smoke_test.py \
    --config configs/cde_ff49.yaml \
    --report-json /kaggle/working/ff49_multiseed_data_check.json
assert _exit_code == 0, "data-smoke failed."

In [ ]:
from diffusion_portfolio.data.preparation import load_standardizer

report = json.loads(Path("/kaggle/working/ff49_multiseed_data_check.json").read_text())
for run in REFERENCES:
    scaler = load_standardizer(run / "standardizer.npz")
    assert scaler.columns == tuple(report["columns"]), "Reference asset labels/order differ from the current panel."
    np.testing.assert_array_equal(scaler.mean, report["standardizer"]["mean"])
    np.testing.assert_array_equal(scaler.std, report["standardizer"]["std"])
    metrics = json.loads((run / "validation_probabilistic_metrics.json").read_text())
    assert metrics["n_validation"] == report["validation_samples"]
    assert [metrics["validation_start"], metrics["validation_end"]] == report["target_ranges"]["validation"]
print("Both seed-42 references match the current data preparation.")

## Additional paired seeds 43 and 44

Seed 42 is reused unchanged. Each additional run uses its matching dataset configuration,
changes only the training seed, and trains for the configured 50 epochs. The runner uses
`--resume` for each child run, skips completed training/validation, and raises on failures.
Rerun this cell after an interruption. No extra seed-42 training is launched.

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/run_multiseed.py \
    --seed42-vanilla-dir /kaggle/working/ff49_vanilla \
    --seed42-cde-dir /kaggle/working/ff49_cde_clean \
    --vanilla-config configs/vanilla_ff49.yaml \
    --cde-config configs/cde_ff49.yaml \
    --output-root /kaggle/working/ff49_multiseed \
    --seeds 43 44 \
    --device cuda
assert _exit_code == 0, "multiseed failed."

## Training and validation curves (also usable after partial training)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, (architecture, seed) in zip(axes.flat, [(a, s) for a in ("vanilla", "cde") for s in (43, 44)]):
    run = ROOT / architecture / f"seed_{seed}"
    if (run / "last.pt").exists():
        history = pd.DataFrame(torch.load(run / "last.pt", map_location="cpu", weights_only=False)["history"])
        history.plot(x="epoch", y=["train_loss", "val_loss"], ax=ax)
        display(history.tail())
    ax.set(title=f"{architecture}, seed {seed}", xlabel="Epoch", ylabel="Diffusion loss")
    ax.grid(True, alpha=.3)
fig.tight_layout()
plt.show()

## Completed validation comparisons

All six runs must be complete. Mean/std use the KF12 sample standard deviation (`ddof=1`);
paired deltas are **CDE minus Vanilla** for each seed. Lower metrics are preferable.

In [ ]:
runs = {("vanilla", 42): Path("/kaggle/working/ff49_vanilla"), ("cde", 42): CDE_DIR}
runs.update({(a, s): ROOT / a / f"seed_{s}" for a in ("vanilla", "cde") for s in (43, 44)})
for run in runs.values():
    require_completed_run(run, assets=49, validation=True)
from diffusion_portfolio.training.multiseed import aggregate, read_result
per_seed, summary = aggregate([read_result(run, architecture=a, seed=s) for (a, s), run in runs.items()])
display(per_seed)
display(pd.concat({a: pd.DataFrame(v).T for a, v in summary["summary"]["architectures"].items()}, names=["Architecture", "Metric"]))
display(pd.DataFrame(summary["paired_rows"]))
display(pd.DataFrame(summary["summary"]["paired"]).T)

## Manual backup

Click only when ready to upload. The latest central dataset is copied in full, then only this
notebook's own namespace is replaced. **Serialize uploads across Kaggle sessions**; there is no
cross-session lock or automatic merge. A failed/empty download stops upload. Prerequisite seed-42
directories are never replaced here. Resumable diffusion checkpoints can be backed up mid-run;
probe/router steps need their completed per-step artifacts because those CLIs are single-shot.

In [ ]:
assert any(ROOT.rglob("last.pt")), "No trained multi-seed checkpoint to back up."

for last in ROOT.rglob("last.pt"):
    check_checkpoint_pair(last.parent)
artifact_path = Path(kagglehub.dataset_download(HANDLE, force_download=True))
if not any(p.is_file() and p.name != ".complete" for p in artifact_path.rglob("*")):
    raise RuntimeError("Empty central artifact dataset; refusing to replace it.")
staging = ROOT.parent / (ROOT.name + "_artifact_sync")
if staging.exists():
    shutil.rmtree(staging)
shutil.copytree(artifact_path, staging)
destination = staging / EXPERIMENT
if destination.exists():
    shutil.rmtree(destination)
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(ROOT, destination)
kagglehub.dataset_upload(HANDLE, str(staging), ignore_patterns=[".complete"],
                         version_notes=f"Available research artifacts: {EXPERIMENT}")
print("Backed up:", EXPERIMENT)